# Petrophysics for LLM agents

A language model can write *about* petrophysics but cannot *do* it: no logs, no solver, no QC.
The useful pattern is the inverse: a deterministic engine computes, and emits **structured,
grounded artifacts** an agent can read and reason over. This library is built for that. Every
run emits a JSON **decision trace**, per-curve **statistics**, and **QC facts**, so the model
answers from numbers the engine produced, not from guesses. It's all library calls.

In [1]:
import sys, json, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
REPO = Path.cwd()
while not (REPO / "petrophys").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from petrophys.io import read_las, export_reference_json_work_package_directory
from petrophys.workflow import run_dynamic_workflow
GEO = REPO / "petrophys" / "data" / "geolink" 

## 1. The decision trace as JSON

`run_dynamic_workflow(...).to_json()` serialises the whole interpretation. An agent reads *why* a value was chosen, not just the value.

In [2]:
well = read_las(GEO / "30_6-5.las")
result = run_dynamic_workflow(well, lithology="clastic")
print(result.to_json(include_curves=["PHIE"])[:700], "...")

{
  "schema": "dynamic_workflow_trace",
  "schema_version": 1,
  "well": {
    "name": "30/6-5",
    "depth_unit": "m",
    "top": 176.60999,
    "bottom": 3556.08032,
    "step": 0.15235000000029686,
    "n_samples": 22176,
    "curve_count": 60,
    "available_curves": [
      "BAD_HOLE",
      "BS",
      "BVW",
      "CALI",
      "CROSSOVER_FLAG",
      "DRHO",
      "DTC",
      "DTE",
      "FZI",
      "GDEN",
      "GR",
      "INVASION_RATIO",
      "KH_CORE",
      "KV_CORE",
      "LITHOLOGY_GEOLINK",
      "MMQC",
      "NET_PAY",
      "NET_RES",
      "NET_ROCK",
      "NPHI",
      "PEF",
      "PERM",
      "PERM_CC",
      "PHIC",
      "PHID",
      "PHIE",
      "PHIE_CC" ...


## 2. Grounded statistics and QC

`export_reference_json_work_package_*` writes a reference JSON per well: per-curve statistics, the lithology distribution, and data-completeness / QC, the factual substrate that keeps an agent honest.

In [3]:
tmp = Path(tempfile.mkdtemp())
export_reference_json_work_package_directory(str(GEO), str(tmp), pattern="30_6-5.las")
ar = json.loads(next(tmp.rglob("30_6-5.json")).read_text(encoding="utf-8"))["analysis_results"]

print("core porosity present:", ar["core_phi_present"], "| missing logs:",
      ar["data_completeness"]["missing_primary_logs"])
stats = ar["statistics"]
pd.DataFrame({k.replace("_overall", ""): v for k, v in stats.items() if k.endswith("_overall")}).T[
    ["minimum", "arithmetic_mean", "median", "geometric_mean", "maximum"]].round(2).head(8)

core porosity present: True | missing logs: ['RXO', 'DTS', 'RMF']


,minimum,arithmetic_mean,median,geometric_mean,maximum
CALI,8.20,15.21,14.38,14.74,24.14
DRHO,-1.00,0.00,-0.00,0.01,0.41
NPHI,0.00,0.39,0.40,0.35,0.60
RHOB,1.00,2.18,2.11,2.16,2.78
PEF,2.46,4.36,4.05,4.24,13.69
GR,5.44,59.78,53.31,53.14,238.00
DTC,34.04,124.31,133.60,120.07,207.36
DTE,32.84,123.29,133.31,118.84,206.22


## 3. Aggregating a portfolio

Agents reason across wells. Running the workflow over the bundled wells yields a tidy table to filter, rank, or flag.

In [4]:
rows = []
for las in sorted(GEO.glob("*.las")):
    w = read_las(las)
    r = run_dynamic_workflow(w, lithology="clastic")
    g = lambda *n: next((np.asarray(w.get(x).data, float) for x in n if w.get(x) is not None), None)
    netpay, phie = g("NET_PAY"), g("PHIE")
    rw = next((s for s in r.steps_executed if s.step == "rw"), None)
    rows.append({"well": w.name or las.stem, "curves": len(w.curves),
                 "rw_regime": rw.parameters.get("rwa_regime") if rw else None,
                 "PHIE_p50": round(float(np.nanmedian(phie)), 3) if phie is not None else None,
                 "net_pay_m": round(float(np.nansum(netpay > 0.5) * (w.step or 0.15)), 1) if netpay is not None else None,
                 "has_core": w.get("PHIC") is not None})
pd.DataFrame(rows)

,well,curves,rw_regime,PHIE_p50,net_pay_m,has_core
0,16/7-1,46,wide_spread_hc_leg_low_tail,0.183,233.8,False
1,29/3-1,49,narrow_spread_all_wet_median,0.030,22.1,False
2,30/6-5,60,narrow_spread_all_wet_median,0.186,30.2,True


## 4. A grounded agent context

Assemble a compact, model-ready context entirely from computed artifacts, no free-text guessing. A retrieval agent attaches this to the user's question so the model answers from the engine's numbers and respects its caveats.

In [5]:
context = {
    "well": well.name,
    "key_statistics": {c.replace("_overall", ""): round(stats[c]["median"], 3)
                       for c in ("PHIE_overall", "VCL_overall", "SW_overall") if c in stats},
    "qc": {"core_calibrated": ar["core_phi_present"],
           "missing_logs": ar["data_completeness"]["missing_primary_logs"],
           "warnings": result.warnings},
}
print(json.dumps(context, indent=2))

{
  "well": "30/6-5",
  "key_statistics": {
    "PHIE": 0.172,
    "VCL": 0.445,
    "SW": 0.911
  },
  "qc": {
    "core_calibrated": true,
    "missing_logs": [
      "RXO",
      "DTS",
      "RMF"
    ],
    "warnings": []
  }
}


**Takeaways**

* The engine is deterministic and auditable; the **LLM is the interface, not the calculator**.
* Three artifact types make a petrophysics agent possible: the **decision trace** (how),
  **statistics** (what), and **QC facts** (how much to trust it), all emitted as JSON by library
  functions.
* The same calls back a directory of wells, so an agent reasons over a **portfolio** and drops
  straight into an MCP / tool-calling server as a petrophysics backend.